In [1]:
import networkx as nx
from collections import defaultdict, deque
import itertools 
from itertools import combinations
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from networkx.algorithms import isomorphism

def shadow(H):
    D_H = set()

    for s in H:
        elements = list(s)
        n = len(elements)

        for j in range(n):
            for k in range(j + 1, n):
                subset = frozenset([elements[j], elements[k]])
                D_H.add(subset)

    return D_H

def binary(n):
    result = set()
    for i in range(n):
        for j in range(i + 1, n):
            result.add(frozenset([i, j]))
    return result


def ternary(n):
    result = set()
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                result.add(frozenset([i, j, k]))
    return result

def get_missing_edge(n,H):
    all_ternary = ternary(n)
    result = []
    for i in all_ternary-H:
        result.append(set(i))
    return result

def get_new_graph(n,H):
    subset = {}

    temp_H = []
    for i in H:
        temp_H.append(set(i))

    missing_edge = get_missing_edge(n,H)
    for i in range(1,len(missing_edge)+1):
        subset[i] = list()

    for i in range(1,len(missing_edge)+1):
        temp = []
        for comb in combinations(missing_edge,i):
            subset[i].append(temp_H + list(comb))

    return subset

def rgraph_to_graph(n, H):
    G = nx.Graph()

    idx = 0
    for edge in H:
        for vtx in edge:
            G.add_edge(vtx, idx + n)
        idx += 1

    return G

def redun_graph(n,H):
    all_new_graph = get_new_graph(n,H)
    graph_level = len(get_missing_edge(n,H))
    result = {}
    for i in range(1,graph_level+1):
        result[i] = list()

    for i in range(1,graph_level+1):
        for graph in all_new_graph[i]:
            if len(result[i]) == 0:
                result[i].append(graph)
            else:
                is_duplicate = False
                for existing in result[i]:
                    temp2 = rgraph_to_graph(n, graph)
                    temp3 = rgraph_to_graph(n, existing)
                    if nx.is_isomorphic(temp2, temp3):
                        is_duplicate = True
                        break
                if not is_duplicate:
                    result[i].append(graph)

    return result

def det_sub(n1,g1,n2,g2):
    if n1 >n2:
        return False
    GM = isomorphism.GraphMatcher(rgraph_to_graph(n2,g2), rgraph_to_graph(n1,g1))
    return GM.subgraph_is_isomorphic()

def get_graph(vtx_num,edge_num):
    
    missing_edge = ternary(vtx_num)
    subset = []
    for comb in combinations(missing_edge,edge_num):
        temp = list(comb)
        subset.append(temp)

    return subset

def redun_edge_graph(vtx_num,edge_num):
    all_new_graph = get_graph(vtx_num,edge_num)
    result = []

    for graph in all_new_graph:
            if len(result) == 0:
                result.append(graph)
            else:
                is_duplicate = False
                for existing in result:
                    temp2 = rgraph_to_graph(vtx_num, graph)
                    temp3 = rgraph_to_graph(vtx_num, existing)
                    if nx.is_isomorphic(temp2, temp3):
                        is_duplicate = True
                        break
                if not is_duplicate:
                    result.append(graph)

    return result

# def 

def autoIsoNum(n,H,G):
    tuples = list(itertools.permutations(range(n),n))
    result = 0
    for i in range(len(tuples)):
        new_graph = [{tuples[i][x] for x in s} for s in G]
        flag = True
        for edge in new_graph:
            if edge not in H:
                flag = False
        if flag:
            result += 1
    return result

def list_autoIsoNum(n,H_list,G):
    result = [0 for i in range(len(H_list))]
    for i in range(len(H_list)):
        result[i] = autoIsoNum(n,H_list[i],G)
    return result


def delIsoSup(n1,underlying,n2,iso_graph):
    subset = {}

    sup_graph = redun_graph(n1,underlying)

    for i in range(1,len(sup_graph)+1):
        subset[i] = list()
    
    for i in range(1,len(sup_graph)+1):
        for graph in sup_graph[i]:
            if not det_sub(n2,iso_graph,n1, graph):
                subset[i].append(graph)

    return subset

def delIsoSup_re(n1,sup_graph,n2,iso_graph):
    subset = {}


    for i in range(1,len(sup_graph)+1):
        subset[i] = list()
    
    for i in range(1,len(sup_graph)+1):
        for graph in sup_graph[i]:
            if not det_sub(n2,iso_graph,n1, graph):
                subset[i].append(graph)

    return subset


K53 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,1,4}), frozenset({0,2,3}), frozenset({0,2,4}), frozenset({0,3,4}), frozenset({1,2,3}), frozenset({1,2,4}), frozenset({1,3,4}), frozenset({2,3,4})}
K43m = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,2,3})}
K43 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,2,3}), frozenset({1,2,3})}
F32 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,1,4}), frozenset({2,3,4})}
F5 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({2,3,4})}
C53m = {frozenset({0,1,2}), frozenset({1,2,3}), frozenset({2,3,4}), frozenset({0,3,4})}
J4 = {frozenset({0, 1, 2}), frozenset({0,1,3}), frozenset({0,1,4}), frozenset({0,2,3}),frozenset({0,2,4}),frozenset({0,3,4})}
C53 = {frozenset({0,1,2}), frozenset({1,2,3}), frozenset({2,3,4}), frozenset({0,3,4}), frozenset({0,1,4})}
J5 = {frozenset({0, 1, 2}), frozenset({0,1,3}), frozenset({0,1,4}), frozenset({0,1,5}),frozenset({0,2,3}),frozenset({0,2,4}),frozenset({0,2,5}),frozenset({0,3,4}),frozenset({0,4,5}),frozenset({0,3,5})}
S4 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,1,4}), frozenset({0,1,5})}
S3 = {frozenset({0,1,2}), frozenset({0,1,3}), frozenset({0,1,4})}
S2 = {frozenset({0,1,2}), frozenset({0,1,3})}
edge= {frozenset({0,1,2})}
empty = set()







In [8]:
sup_graph = redun_graph(4,S2)
all = list()
result=delIsoSup_re(4,sup_graph,5,K53)

for i in range(1,3):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(4,all,S2)

num2 = 0
for i in range(len(all)):
    print(f"S2_{num2} = {int(auto_num[i]/4)} * ThreeGraphTheory(4,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

S2_0 = 3 * ThreeGraphTheory(4,edges =[ [ 0, 1, 3 ], [ 0, 1, 2 ], [ 1, 2, 3 ] ])
S2_1 = 6 * ThreeGraphTheory(4,edges =[ [ 0, 1, 3 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ] ])


In [9]:
sup_graph = redun_graph(5,S3)
all = list()
result=delIsoSup_re(5,sup_graph,5,K53)

for i in range(1,8):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(5,all,S3)

num2 = 0
for i in range(len(all)):
    print(f"S3_{num2} = {int(auto_num[i]/12)} * ThreeGraphTheory(5,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

S3_0 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ] ])
S3_1 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 2, 3, 4 ] ])
S3_2 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 2, 3, 4 ] ])
S3_3 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 2, 3 ] ])
S3_4 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 1, 2, 4 ] ])
S3_5 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 1, 3, 4 ] ])
S3_6 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 2, 3, 4 ], [ 0, 2, 3 ] ])
S3_7 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 2, 3, 4 ], [ 1, 2, 4 ] ])
S3_8 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 2, 3, 4 ], [ 1, 3, 4 ] ])
S3_9 = 4 * ThreeGraphTheory(5,edg

In [10]:
sup_graph = redun_graph(5,C53m)
all = list()
result=delIsoSup_re(5,sup_graph,5,K53)

for i in range(1,7):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(5,all,C53m)

num2 = 0
for i in range(len(all)):
    print(f"C53m_{num2} = {int(auto_num[i]/2)} * ThreeGraphTheory(5,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

C53m_0 = 5 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 1, 4 ] ])
C53m_1 = 1 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 2, 3 ] ])
C53m_2 = 2 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 1, 2, 4 ] ])
C53m_3 = 2 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 2, 4 ] ])
C53m_4 = 7 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 1, 4 ], [ 0, 2, 3 ] ])
C53m_5 = 4 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 2, 3 ], [ 1, 2, 4 ] ])
C53m_6 = 2 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 0, 2, 3 ], [ 0, 2, 4 ] ])
C53m_7 = 4 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0, 3, 4 ], [ 1, 2, 4 ], [ 1, 3, 4 ] ])
C53m_8 = 6 * ThreeGraphTheory(5,edges =[ [ 1, 2, 3 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 0,

In [14]:
sup_graph = redun_graph(5,J4)
all = list()
result=delIsoSup_re(5,sup_graph,5,K53)

for i in range(1,5):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(5,all,J4)

num2 = 0
for i in range(len(all)):
    print(f"J4_{num2} = {int(auto_num[i]/24)} * ThreeGraphTheory(5,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

J4_0 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 0, 2, 3 ], [ 0, 1, 2 ], [ 0, 2, 4 ], [ 0, 1, 3 ], [ 1, 2, 3 ] ])
J4_1 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 0, 2, 3 ], [ 0, 1, 2 ], [ 0, 2, 4 ], [ 0, 1, 3 ], [ 1, 2, 3 ], [ 2, 3, 4 ] ])
J4_2 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 0, 2, 3 ], [ 0, 1, 2 ], [ 0, 2, 4 ], [ 0, 1, 3 ], [ 1, 2, 3 ], [ 2, 3, 4 ], [ 1, 3, 4 ] ])


In [7]:
sup_graph = redun_graph(6,S4)
all = list()
result=delIsoSup_re(6,sup_graph,5,K53)

for i in range(1,17):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(6,all,S4)

num2 = 0
for i in range(len(all)):
    print(f"S4_{num2} = {int(auto_num[i]/48)} * ThreeGraphTheory(6,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

S4_0 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ] ])
S4_1 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 2, 3, 4 ] ])
S4_2 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 1, 3, 5 ] ])
S4_3 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 2, 3, 4 ] ])
S4_4 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 0, 2, 5 ] ])
S4_5 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 1, 2, 5 ] ])
S4_6 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 0, 2, 4 ] ])
S4_7 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2 ], [ 0, 1, 5 ], [ 0, 3, 4 ], [ 2, 4, 5 ] ])
S4_8 = 1 * ThreeGraphTheory(6,edges =[ [ 0, 1, 4 ], [ 0, 1, 3 ], [ 0, 1, 2

In [13]:
sup_graph = redun_graph(5,C53)
all = list()
result=delIsoSup_re(5,sup_graph,5,K53)

for i in range(1,6):
    for graph in result[i]:
        all.append(graph)

auto_num = list_autoIsoNum(5,all,C53)

num2 = 0
for i in range(len(all)):
    print(f"C53_{num2} = {int(auto_num[i]/10)} * ThreeGraphTheory(5,edges =[",end=" ")
    num = 0
    for edge in all[i]:
        print("[", end=" ")
        print(", ".join(str(val) for val in edge), end=" ") 
        if num == len(all[i]) -1:
            print("]", end=" ")
        else:
            print("],", end=" ")
            num += 1
    num2 += 1
    print("])")

C53_0 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ] ])
C53_1 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ], [ 1, 2, 4 ] ])
C53_2 = 1 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ], [ 0, 2, 4 ] ])
C53_3 = 2 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ], [ 1, 2, 4 ], [ 0, 2, 4 ] ])
C53_4 = 4 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ], [ 1, 2, 4 ], [ 0, 1, 3 ] ])
C53_5 = 6 * ThreeGraphTheory(5,edges =[ [ 0, 3, 4 ], [ 0, 1, 4 ], [ 2, 3, 4 ], [ 0, 1, 2 ], [ 1, 2, 3 ], [ 0, 2, 3 ], [ 1, 2, 4 ], [ 0, 2, 4 ], [ 0, 1, 3 ] ])
